# Demand Time Series and Forecasting

Exploration, decomposition and forecasting of daily demand for a digital health service, using a session log (one row per session, with a provider ID, a user ID and a date).

**All data in this notebook are simulated.** Variable names, volumes, dates, calendars and parameter values are invented for demonstration and do not correspond to any real dataset or service.

## 1. Data

### Imports and simulated session log

In [ ]:
import itertools
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import norm
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from prophet import Prophet
from prophet.plot import add_changepoints_to_plot
from prophet.diagnostics import cross_validation, performance_metrics

logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").setLevel(logging.WARNING)

In [ ]:
def simulate_sessions(start="2020-01-06", n_days=1400, seed=7):
    """Simulated session log: one row per session (date, provider_id, user_id).

    Daily demand = growing trend x weekly pattern x yearly pattern x event dips + noise.
    """
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, periods=n_days, freq="D")
    t = np.arange(n_days)

    trend = 40 + 0.12 * t
    weekly = np.array([1.15, 1.2, 1.15, 1.1, 1.0, 0.45, 0.2])[dates.day_of_week]
    yearly = 1 + 0.25 * np.sin(2 * np.pi * (dates.day_of_year - 30) / 365.25) \
               + 0.10 * np.sin(4 * np.pi * dates.day_of_year / 365.25)
    lam = trend * weekly * yearly
    in_break = np.isin(dates.normalize(), EVENTS["ds"].values)
    lam = np.where(in_break, lam * 0.3, lam)
    counts = rng.poisson(lam)

    n_providers, n_users = 600, 25000
    provider_start = rng.integers(0, n_days, n_providers)
    rows = []
    for day, c in zip(range(n_days), counts):
        active = np.flatnonzero(provider_start <= day + 30)
        prov = rng.choice(active, c)
        user = np.minimum(rng.zipf(1.6, c) + (day * 15), n_users * 3)
        rows.append(pd.DataFrame({"date": dates[day], "provider_id": prov, "user_id": user}))
    return pd.concat(rows, ignore_index=True)


# Simulated calendar events (generic "low-demand" periods) used as holidays
EVENTS = pd.DataFrame({
    "holiday": "break",
    "ds": pd.to_datetime(np.concatenate([
        pd.date_range(f"{y}-12-22", f"{y + 1}-01-02").values for y in range(2019, 2025)
    ])),
    "lower_window": 0,
    "upper_window": 0,
})

sess = simulate_sessions()
sess.head()

### Aggregation

Incomplete recent periods are trimmed, then sessions are aggregated to daily, weekly and monthly counts.

In [ ]:
cutoff = sess["date"].max() - pd.Timedelta(days=14)
sess = sess[sess["date"] <= cutoff]

sessions_per_day = sess.groupby("date").size().to_frame("session_count")
sessions_per_week = sessions_per_day.resample("W").sum()
sessions_per_month = sessions_per_day.resample("ME").sum()

## 2. Exploratory visualisation

In [ ]:
def bar_plot(series, title, ylabel, width=1.0, fname=None):
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.bar(series.index, series.values, width=width, color="#1E40FF")
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel(ylabel)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    if fname:
        plt.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()


bar_plot(sessions_per_day["session_count"], "Sessions per day", "Sessions")
bar_plot(sessions_per_week["session_count"], "Sessions per week", "Sessions", width=5)
bar_plot(sessions_per_month["session_count"], "Sessions per month", "Sessions", width=20)

Daily sessions by day type:

In [ ]:
dow = sessions_per_day.index.day_of_week
fig, ax = plt.subplots(figsize=(12, 4.5))
for label, mask in [("Weekdays", dow < 5), ("Saturdays", dow == 5), ("Sundays", dow == 6)]:
    d = sessions_per_day[mask]
    ax.plot(d.index, d["session_count"], marker="o", markersize=2, linewidth=1, label=label)
ax.set_title("Sessions by day type")
ax.set_ylabel("Sessions")
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

### New providers and new users

The first appearance of each provider and each user in the log gives the number of newcomers per day and per week.

In [ ]:
new_providers = sess.groupby("provider_id")["date"].min().value_counts().sort_index().rename("new_providers")
new_users = sess.groupby("user_id")["date"].min().value_counts().sort_index().rename("new_users")

weekly_summary = pd.concat([
    sessions_per_week["session_count"],
    new_users.resample("W").sum(),
    new_providers.resample("W").sum(),
], axis=1).fillna(0)

fig, ax = plt.subplots(figsize=(12, 5))
for col, color in [("session_count", "mediumseagreen"), ("new_users", "darkorange"), ("new_providers", "royalblue")]:
    ax.plot(weekly_summary.index, weekly_summary[col], marker="o", markersize=3, label=col, color=color)
ax.set_yscale("log")
ax.set_title("Weekly sessions, new users and new providers")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Seasonal-trend decomposition

Weekly counts are decomposed with STL (period = 52 weeks), plus two multiplicative variants: classical multiplicative decomposition, and robust STL on log counts back-transformed to the original scale.

In [ ]:
weekly = sessions_per_week["session_count"]

STL(weekly, period=52).fit().plot()
plt.suptitle("STL decomposition of weekly sessions", y=1.02)
plt.show()

decomp_mult = seasonal_decompose(weekly, model="multiplicative", period=52)
decomp_mult.plot()
plt.suptitle("Multiplicative decomposition of weekly sessions", y=1.02)
plt.show()

In [ ]:
res = STL(np.log(weekly), seasonal=53, robust=True).fit()
components = {"Observed": weekly, "Trend": np.exp(res.trend),
              "Seasonal": np.exp(res.seasonal), "Residual": np.exp(res.resid)}

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, (name, series) in zip(axes, components.items()):
    ax.plot(series)
    ax.set_ylabel(name)
plt.suptitle("Robust log-STL (multiplicative scale)")
plt.tight_layout()
plt.show()

One yearly cycle of the seasonal component, overlaid with the simulated low-demand periods:

In [ ]:
seasonal_cycle = decomp_mult.seasonal.iloc[:52]
cycle_events = EVENTS[(EVENTS["ds"] >= seasonal_cycle.index.min()) & (EVENTS["ds"] <= seasonal_cycle.index.max())]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(seasonal_cycle.index, seasonal_cycle.values, marker="o", label="Seasonal component")
if not cycle_events.empty:
    ax.axvspan(cycle_events["ds"].min(), cycle_events["ds"].max(), color="lightblue", alpha=0.4, label="Break")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_title("Seasonal component (one yearly cycle)")
ax.set_ylabel("Seasonal factor")
ax.grid(True)
ax.legend()
plt.show()

## 4. Forecasting with Prophet

Prophet is a generalised additive model with a piecewise-linear trend, Fourier-based weekly and yearly seasonality, and holiday effects (here, the simulated calendar events).

### Baseline model

Multiplicative seasonality on the raw daily counts, with a 6-month forecast:

In [ ]:
df_prophet = sessions_per_day.reset_index().rename(columns={"date": "ds", "session_count": "y"})

baseline = Prophet(weekly_seasonality=True, yearly_seasonality=True,
                   seasonality_mode="multiplicative", holidays=EVENTS)
baseline.fit(df_prophet)

future = baseline.make_future_dataframe(periods=182, freq="D")
forecast = baseline.predict(future)
for col in ["yhat", "yhat_lower", "yhat_upper"]:
    forecast[col] = forecast[col].clip(lower=0)

fig = baseline.plot(forecast)
add_changepoints_to_plot(fig.gca(), baseline, forecast)
baseline.plot_components(forecast)
plt.show()

In [ ]:
df_cv = cross_validation(baseline, initial="730 days", period="60 days", horizon="90 days")
performance_metrics(df_cv, rolling_window=1)

### Hyperparameter tuning

Grid search over the prior scales on the `log1p`-transformed series (which prevents negative forecasts), each combination scored by rolling-origin cross-validation MAPE on the original scale:

In [ ]:
df_log = df_prophet.assign(y=np.log1p(df_prophet["y"]))

param_grid = {
    "seasonality_prior_scale": [0.1, 10.0],
    "changepoint_prior_scale": [0.01, 0.5],
    "holidays_prior_scale": [0.1, 10.0],
}
all_params = [dict(zip(param_grid, v)) for v in itertools.product(*param_grid.values())]

mapes = []
for params in all_params:
    m = Prophet(seasonality_mode="additive", holidays=EVENTS, **params).fit(df_log)
    cv = cross_validation(m, initial="730 days", period="90 days", horizon="90 days")
    cv[["y", "yhat"]] = np.expm1(cv[["y", "yhat"]])
    mapes.append(performance_metrics(cv, rolling_window=1)["mape"].iloc[0])

tuning_results = pd.DataFrame(all_params).assign(mape=mapes).sort_values("mape")
best_params = all_params[int(np.argmin(mapes))]
tuning_results

### Final model

In [ ]:
model = Prophet(weekly_seasonality=True, seasonality_mode="additive", holidays=EVENTS, **best_params)
model.fit(df_log)

future = model.make_future_dataframe(periods=365, freq="D")
forecast = model.predict(future)

fig = model.plot(forecast)
add_changepoints_to_plot(fig.gca(), model, forecast)
model.plot_components(forecast)
plt.show()

### Validation with bias correction

Back-transforming a log-scale forecast with `expm1` gives the median, not the mean. Assuming log-normal errors, σ is recovered from the prediction interval and the mean is corrected as exp(μ + σ²/2) − 1.

In [ ]:
def back_transform(cv, interval_width):
    cv = cv.copy()
    z = norm.ppf((1 + interval_width) / 2)
    mu = cv["yhat"].to_numpy()
    sigma = np.maximum((cv["yhat_upper"].to_numpy() - mu) / z, 1e-8)
    cv["y"] = np.expm1(cv["y"])
    cv["yhat"] = np.expm1(mu + 0.5 * sigma**2)
    cv["yhat_lower"] = np.expm1(mu - z * sigma)
    cv["yhat_upper"] = np.expm1(mu + z * sigma)
    return cv


for horizon in ["30 days", "90 days", "182 days"]:
    cv = back_transform(cross_validation(model, initial="730 days", period="60 days", horizon=horizon),
                        model.interval_width)
    print(f"Horizon {horizon}:")
    print(performance_metrics(cv, rolling_window=1)[["horizon", "rmse", "mae", "mape", "coverage"]], "\n")

## 5. Capacity, saturation and cost

**Daily carrying capacity** is the number of providers active in a rolling window multiplied by the mean number of sessions per provider per day. The **saturation date** is the first forecast day on which expected demand exceeds that capacity, assuming no new providers join. **Expected expenditure** applies a per-session cost to the forecast.

In [ ]:
WINDOW_DAYS = 60
COST_PER_SESSION = 50.0   # illustrative unit cost

daily_load = sess.groupby(["provider_id", "date"]).size().rename("daily_sessions").reset_index()
provider_stats = daily_load.groupby("provider_id")["daily_sessions"].agg(["min", "max", "mean"])
print(provider_stats.describe().round(2))
mean_load = provider_stats["mean"].mean()

first_last = sess.groupby("provider_id")["date"].agg(["min", "max"])
days = sessions_per_day.index
active = [((first_last["min"] <= d) & (first_last["max"] >= d - pd.Timedelta(days=WINDOW_DAYS))).sum() for d in days]
sessions_per_day["active_providers"] = active
sessions_per_day["cap"] = sessions_per_day["active_providers"] * mean_load

bar_plot(sessions_per_day["cap"], "Estimated daily carrying capacity", "Sessions / day")

In [ ]:
capacity_now = sessions_per_day["cap"].iloc[-1]
fc = forecast[forecast["ds"] > sessions_per_day.index.max()].copy()
fc["expected"] = np.expm1(fc["yhat"])

over = fc[fc["expected"] > capacity_now]
saturation = over["ds"].min() if not over.empty else None
print(f"Current capacity: {capacity_now:.0f} sessions/day")
print("Saturation date:", saturation.date() if saturation is not None else "not reached within horizon")
print(f"Expected expenditure over the horizon: {fc['expected'].sum() * COST_PER_SESSION:,.0f}")